# Getting Started with PFLARE

This notebook introduces PFLARE, a library that adds two new preconditioner (PC) types to PETSc:

- `PCPFLAREINV`: a collection of approximate inverses — useful as a smoothers in multigrid methods or as preconditioners
- `PCAIR`: a reduction multigrid hierarchy using AIR (approximate ideal restriction)

We will:
1. Register the PFLARE PC types with PETSc
2. Assemble a simple 2D Laplacian (an SPD problem, just to verify the setup)
3. Solve it with `PCPFLAREINV` and with `PCAIR`, comparing convergence histories
4. Print hierarchy statistics

**Prerequisites**: `petsc4py` and `PFLARE` must be installed and importable. These notebooks use Python to demonstrate, but PFLARE can be used easily from C/C++/Fortran.

## 1. Import and register PFLARE

Importing `pflare` automatically calls `PCRegister_PFLARE()`, which makes the `air` and `pflareinv` PC types available to PETSc.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare  # registers 'air' and 'pflareinv' PC types

print("PETSc version:", PETSc.Sys.getVersion())
print("PFLARE imported successfully")

## 2. Assemble a 2D Laplacian

We build a standard 5-point finite-difference Laplacian on an $N \times N$ grid.  This is a symmetric positive definite (SPD) matrix — easy for any solver.  We use this first just to verify that PFLARE is working correctly.

In later notebooks we will switch to **advection-dominated** problems where AIRG really shows its strength.

In [ ]:
def build_2d_laplacian(N):
    """5-point FD Laplacian on an N×N grid. Returns (A, b, x)."""
    n = N * N
    A = PETSc.Mat().create()
    A.setSizes([n, n])
    A.setFromOptions()
    A.setPreallocationNNZ(5)
    A.setUp()

    h2 = 1.0 / ((N + 1) ** 2)  # (1/h)^2 factor absorbed into stencil
    rstart, rend = A.getOwnershipRange()

    for row in range(rstart, rend):
        i, j = divmod(row, N)
        A.setValue(row, row, 4.0)
        if j > 0:     A.setValue(row, row - 1, -1.0)
        if j < N - 1: A.setValue(row, row + 1, -1.0)
        if i > 0:     A.setValue(row, row - N, -1.0)
        if i < N - 1: A.setValue(row, row + N, -1.0)

    A.assemblyBegin()
    A.assemblyEnd()

    b = A.createVecRight()
    b.set(1.0)
    x = A.createVecRight()
    x.set(0.0)
    return A, b, x

N = 50  # 50×50 = 2500 unknowns
A, b, x = build_2d_laplacian(N)
print(f"Matrix size: {A.getSize()[0]} × {A.getSize()[1]}")
print(f"Number of non-zeros: {A.getInfo()['nz_used']:.0f}")

## 3. Solve with PCPFLAREINV

`PCPFLAREINV` by default applies an assembled, sparsity-controlled GMRES polynomial. It forms an approximate inverse $p_k(A) \approx A^{-1}$. There are many different types of approximate inverse available in PCPFLAREINV.

These are useful as standalone preconditioners, or as smoothers within the `PCAIR` multigrid hierarchy.

We use GMRES as the outer Krylov solver and collect residual norms to plot convergence.

In [ ]:
def solve_and_collect(A, b, pc_type, pc_options=None, x0=None):
    """Solve A x = b with the given PC type and return residual history."""
    opts = PETSc.Options()
    if pc_options:
        for k, v in pc_options.items():
            opts[k] = v

    ksp = PETSc.KSP().create()
    ksp.setOperators(A)
    ksp.setType('gmres')
    ksp.setTolerances(rtol=1e-10, max_it=200)

    pc = ksp.getPC()
    pc.setType(pc_type)

    residuals = []
    def monitor(ksp, it, rnorm):
        residuals.append(rnorm)
    ksp.setMonitor(monitor)

    ksp.setFromOptions()

    sol = A.createVecRight()
    if x0 is not None:
        sol.copy(x0)
    ksp.solve(b, sol)

    reason = ksp.getConvergedReason()
    print(f"  PC={pc_type:12s}  iters={ksp.getIterationNumber():4d}  reason={reason}")
    return np.array(residuals)

# -- pflareinv: by default a 6th GMRES polynomial
r_pflareinv = solve_and_collect(
    A, b,
    pc_type='pflareinv',
)

## 4. Solve with PCAIR (AIRG multigrid)

`PCAIR` uses a reduction multigrid. We will discuss the default options in a later notebook, but for now we enable `-pc_air_print_stats_timings` to see the hierarchy structure.

In [ ]:
# -- air: AIRG multigrid
r_air = solve_and_collect(
    A, b,
    pc_type='air',
    pc_options={
        'pc_air_print_stats_timings': True,
    }
)

## 5. Also try plain GMRES (no preconditioning) as a baseline

In [ ]:
r_none = solve_and_collect(A, b, pc_type='none')

## 6. Compare convergence histories

In [ ]:
# Ensure required imports are present in this cell in case earlier cells
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

# Safely retrieve histories if earlier cells didn't run
r_none = globals().get('r_none', np.array([]))
r_pflareinv = globals().get('r_pflareinv', np.array([]))
r_air = globals().get('r_air', np.array([]))

if r_none.size > 0:
    ax.semilogy(r_none / r_none[0], 'k--', label='No preconditioner')
if r_pflareinv.size > 0:
    ax.semilogy(r_pflareinv / r_pflareinv[0], 'b-o', markersize=3, label='PCPFLAREINV (6th order)')
if r_air.size > 0:
    ax.semilogy(r_air / r_air[0], 'r-s', markersize=3, label='PCAIR / AIRG')

ax.set_xlabel('GMRES iteration')
ax.set_ylabel('Relative residual')
ax.set_title(f'2D Laplacian ({N}×{N} grid, {N*N} unknowns)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('convergence_laplacian.png', dpi=120)
plt.show()
print("Saved: convergence_laplacian.png")

## 7. What's next?

The Laplacian is SPD — almost any preconditioner will work. AIR methods shows their advantages on **strongly advective** and asymmetric problems. 

In order to understand how AIR methods work, we must first begin by discussing some of the approximate inverses found in `PCPFLAREINV`.

**[02_pcpflareinv.ipynb](02_pcpflareinv.ipynb)**: Examine some of the approximate inverses found in PCPFLAREINV.